# BankEff_R — Solution

Banking adaptation of FuelEcon_R / Cookbook Ch. 2.  
`data/banks.csv`: **2,550 rows**, years **1990–2024**, **54 banks**. Teaching panel — not live Call Reports.

Reference: `bankeff_r_charts.png`, `bankeff_r_banks.png`, `bankeff_r_flowchart.png`.


## 0. Setup


In [ ]:
library(ggplot2)
library(dplyr)
library(tidyr)
library(readr)
theme_set(theme_minimal(base_size = 12))
options(dplyr.summarise.inform = FALSE)


## 1. Import


In [ ]:
# banks <- read.csv("data/banks.csv", stringsAsFactors = FALSE)
banks <- read_csv("data/banks.csv", show_col_types = FALSE)
head(banks, 3)
cat("nrow =", nrow(banks), " ncol =", ncol(banks), "\n")
print(names(banks))


## 2. Years, charters, flag types


In [ ]:
cat("unique years:", length(unique(banks$year)),
    " from", min(banks$year), "to", max(banks$year), "\n")
print(table(banks$charter, useNA = "ifany"))

cat("\nclass sFlag:", class(banks$sFlag), " unique:", paste(unique(banks$sFlag), collapse = " | "), "\n")
cat("class tFlag:", class(banks$tFlag), " unique:", paste(unique(banks$tFlag), collapse = " | "), "\n")

banks <- banks %>%
  mutate(digital_flag = ifelse(!is.na(tFlag) & tFlag %in% c("T", "TRUE", TRUE), "Digital flag", "No flag"))
print(with(banks, table(digital_flag, decade = 10 * (year %/% 10))))


## 3. Funding


In [ ]:
banks$funding[banks$funding == ""] <- NA
banks$funding2 <- banks$funding
print(table(banks$funding2, useNA = "ifany"))


## 4. All-institution mean ROA

Post-2014 lift mixes Fintech / ILC (and some investment banks) into the same column as community ROA.


In [ ]:
roaByYr <- banks %>%
  group_by(year) %>%
  summarise(
    avgROA = mean(roa, na.rm = TRUE),
    avgNIM = mean(nim, na.rm = TRUE),
    avgEff = mean(efficiency, na.rm = TRUE),
    n = n()
  )

ggplot(roaByYr, aes(year, avgROA)) +
  geom_point(color = "#1f4e79") +
  geom_smooth(se = TRUE, color = "#c0392b") +
  geom_vline(xintercept = 2008, linetype = "dashed", color = "grey50") +
  labs(x = "Year", y = "Average ROA (%)",
       title = "All institutions — mean ROA",
       subtitle = "Includes Fintech / ILC and investment banks. Do not brief this as community-bank profitability.")


## 5. Traditional commercial only


In [ ]:
tradBanks <- banks %>%
  filter(
    charter %in% c("Community Commercial", "Regional Commercial", "Money Center"),
    is.na(charter2) | charter2 == "",
    is.na(atvType) | !(atvType %in% c("Digital-only"))
  )

roaByYr_Trad <- tradBanks %>%
  group_by(year) %>%
  summarise(avgROA = mean(roa, na.rm = TRUE), n = n())

ggplot(roaByYr_Trad, aes(year, avgROA)) +
  geom_point(color = "#1f4e79") +
  geom_smooth(se = TRUE, color = "#c0392b") +
  geom_vline(xintercept = 2008, linetype = "dashed", color = "grey50") +
  labs(x = "Year", y = "Average ROA (%)",
       title = "Traditional commercial banks only",
       subtitle = "2008–09 collapse, partial recovery — not the all-file 2024 print")

compare <- bind_rows(
  roaByYr %>% transmute(year, avgROA, series = "All institutions"),
  roaByYr_Trad %>% transmute(year, avgROA, series = "Traditional commercial")
)
ggplot(compare, aes(year, avgROA, color = series)) +
  geom_line(linewidth = 1) + geom_point(size = 1.4) +
  scale_color_manual(values = c("All institutions" = "#c0392b",
                                "Traditional commercial" = "#1f4e79")) +
  labs(x = "Year", y = "Mean ROA (%)", color = NULL,
       title = "Why the all-institution line misleads after ~2014")


## 6. Assets vs ROA


In [ ]:
tradBanks$assets_b <- as.numeric(tradBanks$assets_b)

ggplot(tradBanks, aes(log10(pmax(assets_b, 0.05)), roa)) +
  geom_point(alpha = 0.18, size = 0.8, color = "#2e86ab") +
  geom_smooth(color = "#c0392b") +
  labs(x = "log10 assets ($bn)", y = "ROA (%)",
       title = "Larger books → slightly lower ROA (traditional commercial)")

avgSize <- tradBanks %>%
  group_by(year) %>%
  summarise(avgAssets = mean(assets_b, na.rm = TRUE))

ggplot(avgSize, aes(year, avgAssets)) +
  geom_point(color = "#6c3483") + geom_smooth(color = "#c0392b") +
  geom_vline(xintercept = 2008, linetype = "dashed", color = "grey50") +
  labs(x = "Year", y = "Average assets ($bn)",
       title = "Average reported assets (mix + consolidation)")

byYear <- tradBanks %>%
  group_by(year) %>%
  summarise(
    `Average ROA` = mean(roa, na.rm = TRUE),
    `Avg assets ($bn)` = mean(assets_b, na.rm = TRUE)
  )
byYear2 <- byYear %>% pivot_longer(-year, names_to = "variable", values_to = "value")

ggplot(byYear2, aes(year, value)) +
  geom_point() + geom_smooth() +
  facet_wrap(~variable, ncol = 1, scales = "free_y") +
  labs(x = "Year", y = NULL,
       title = "Traditional commercial: ROA vs asset size on aligned years")


## 7. Community banks


In [ ]:
commBanks <- tradBanks %>% filter(charter == "Community Commercial")

ggplot(commBanks %>% filter(!is.na(funding2)),
       aes(factor(year), roa)) +
  geom_boxplot(outlier.size = 0.3, fill = "#d6eaf8") +
  facet_wrap(~funding2) +
  theme(axis.text.x = element_text(angle = 90, vjust = 0.5, size = 6)) +
  labs(x = "Year", y = "ROA (%)",
       title = "Community-bank ROA by year and funding mix")

ggplot(commBanks %>% filter(!is.na(funding2)),
       aes(factor(year), fill = factor(funding2))) +
  geom_bar(position = "fill") +
  geom_hline(yintercept = 0.5, linetype = 2) +
  theme(axis.text.x = element_text(angle = 45, hjust = 1, size = 7)) +
  scale_fill_manual(values = c(Core = "#1f4e79", Wholesale = "#f4a261")) +
  labs(x = "Year", y = "Proportion of community rows",
       fill = "Funding",
       title = "Core vs wholesale funding — community catalog")


## 8. Names present every year 1990–2014


In [ ]:
bankCount <- commBanks %>%
  group_by(year) %>%
  summarise(nBanks = n_distinct(bank))

ggplot(bankCount, aes(year, nBanks)) +
  geom_point() + geom_line() +
  labs(x = "Year", y = "Unique community banks",
       title = "Community names offered in the panel each year")

# Use traditional commercial for the 12-name continuous list
uniqBanks <- split(tradBanks$bank[tradBanks$year <= 2014],
                   tradBanks$year[tradBanks$year <= 2014])
uniqBanks <- lapply(uniqBanks, unique)
commonBanks <- Reduce(intersect, uniqBanks)
print(sort(commonBanks))
cat("n common 1990-2014:", length(commonBanks), "\n")

avg_common <- tradBanks %>%
  filter(bank %in% commonBanks, year <= 2024) %>%
  group_by(year, bank) %>%
  summarise(avgROA = mean(roa, na.rm = TRUE))

ggplot(avg_common, aes(year, avgROA)) +
  geom_line(color = "#1f4e79") +
  facet_wrap(~bank, nrow = 3) +
  geom_vline(xintercept = 2008, linetype = "dotted", color = "grey50") +
  labs(x = "Year", y = "Mean ROA (%)",
       title = "Banks present every year 1990–2014 (traditional commercial)")


## Alternate code


In [ ]:
roa_base <- aggregate(roa ~ year, data = banks, FUN = mean)
names(roa_base)[2] <- "avgROA"
head(roa_base)

roa_tapply <- tapply(banks$roa, banks$year, mean)
head(roa_tapply)

# plyr::ddply(banks, ~year, plyr::summarise, avgROA = mean(roa))

ggplot(tradBanks, aes(year, roa)) +
  stat_summary(fun = mean, geom = "point") +
  stat_summary(fun = mean, geom = "line") +
  labs(title = "Alternate: stat_summary on raw rows", y = "Mean ROA")


## More practice — worked


In [ ]:
nim_eff <- tradBanks %>%
  group_by(year) %>%
  summarise(NIM = mean(nim, na.rm = TRUE),
            Efficiency = mean(efficiency, na.rm = TRUE)) %>%
  pivot_longer(-year, names_to = "metric", values_to = "value")
ggplot(nim_eff, aes(year, value, color = metric)) +
  geom_line() +
  labs(title = "Practice 1 — NIM vs efficiency (traditional)", y = NULL)

snap <- commBanks %>% filter(year %in% c(2006, 2009, 2024))
ggplot(snap, aes(factor(year), roa, fill = factor(year))) +
  geom_violin(trim = FALSE, alpha = 0.7) +
  geom_boxplot(width = 0.12, outlier.shape = NA) +
  labs(title = "Practice 2 — community ROA snapshots", x = "Year", fill = NULL)

gain <- banks %>%
  filter(year %in% c(2008, 2024)) %>%
  group_by(book, year) %>%
  summarise(m = mean(roa, na.rm = TRUE), n = n()) %>%
  tidyr::pivot_wider(names_from = year, values_from = c(m, n)) %>%
  mutate(delta = `m_2024` - `m_2008`) %>%
  arrange(desc(delta))
print(gain)

banks %>%
  filter(year %in% c(2009, 2024)) %>%
  group_by(charter, year) %>%
  summarise(mean_npl = mean(npl, na.rm = TRUE), n = n()) %>%
  print()

roa_window <- tradBanks %>%
  filter(year <= 2014) %>%
  group_by(year) %>%
  summarise(avgROA = mean(roa, na.rm = TRUE))
ggplot(roa_window, aes(year, avgROA)) +
  geom_point() + geom_smooth() +
  labs(title = "Practice 5 — traditional ROA, 1990–2014 window")


## Simulation


In [ ]:
year_max <- 2024
include_fintech <- FALSE
noise_sd <- 0
sample_frac <- 1.0
set.seed(42)

sim <- banks
if (sample_frac < 1) sim <- sim %>% slice_sample(prop = sample_frac)
if (noise_sd > 0) sim$roa <- sim$roa + rnorm(nrow(sim), 0, noise_sd)
sim <- sim %>% filter(year <= year_max)

all_s <- sim %>% group_by(year) %>%
  summarise(avgROA = mean(roa, na.rm = TRUE), series = "All institutions")

core_s <- sim %>%
  filter(charter %in% c("Community Commercial", "Regional Commercial", "Money Center"),
         is.na(charter2) | charter2 == "",
         is.na(atvType) | atvType != "Digital-only")
if (include_fintech) {
  extra <- sim %>% filter(charter == "Fintech / ILC")
  core_s <- bind_rows(core_s, extra)
}
core_s <- core_s %>% group_by(year) %>%
  summarise(avgROA = mean(roa, na.rm = TRUE), series = "Traditional (+ fintech if flagged)")

both <- bind_rows(all_s, core_s)
ggplot(both, aes(year, avgROA, color = series)) +
  geom_line(linewidth = 1) +
  labs(title = sprintf("Simulation  year_max=%s  fintech_in_core=%s  noise=%.1f  frac=%.2f",
                       year_max, include_fintech, noise_sd, sample_frac),
       y = "Mean ROA (%)", color = NULL)

last_all <- tail(all_s$avgROA, 1)
last_core <- tail(core_s$avgROA, 1)
post <- core_s %>% filter(year >= 2011, avgROA > 1.10)
hit <- if (nrow(post)) min(post$year) else NA
cat(sprintf("Latest all = %.3f | core = %.3f | first post-crisis core ROA > 1.10 = %s\n",
            last_all, last_core, hit))


## Audience rewrite — worked example

**1. Analyst.** Pooled ROA in this panel is ~1.07% (1990), ~0.48% (2008–09), ~1.36% (2014), ~1.94% (2024). The 2024 print is not a traditional-bank result: Fintech / ILC rows carry ROA near 2–3% and enter after 2014. Restrict to Community + Regional + Money Center, empty `charter2`, not Digital-only: 1.05% → 0.35–0.42% in the crisis → ~1.14% (2014) → ~1.29% (2024). log-assets vs ROA is mildly negative. Treat `sFlag`/`tFlag` as labels. 2024 is a catalog year, not a complete industry census.

**2. ALM / funding technician.** Community rows still fund mostly with core deposits; wholesale share dips after 2008 then creeps back. Spec liquidity coverage off the community mix, not the fintech catalog. NIM compressed after 2020 in the traditional book — do not use the all-file NIM.

**3. CFO / board.** Do not brief “the industry is at ~1.9% ROA” unless the portfolio mix matches the file mix. For a commercial-bank franchise the honest number is low-1%s after a 2008 hole. A target of “match 2014 traditional ROA + 15 bp” is inside the historical residual; a target of 1.9% is a fintech-mix target.

**4. Nonspecialist.** Banks in this list earned about one dollar per hundred dollars of assets in the 1990s, almost lost that in 2008–09, and traditional banks only got part-way back. The higher recent average counts newer digital banks too. Three levers: stay out of a crisis year, don’t only look at giant books, and don’t mix digital startups into a community-bank average.


## Key numbers (this panel)

| Slice | Result |
|---|---|
| Shape | 2,550 × 16 |
| Years | 1990–2024 |
| Charters | Community 941 · Regional 476 · Fintech/ILC 420 · CU 306 · Money Center 223 · IB 184 |
| All-in mean ROA | 1990: 1.07 · 2009: 0.48 · 2014: 1.36 · 2024: 1.94 |
| Traditional commercial ROA | 1990: 1.05 · 2009: 0.35 · 2014: 1.14 · 2024: 1.29 |
| Common names 1990–2014 | 12 continuous commercial banks |

**Takeaway:** Split the charter mix before celebrating an industry ROA miracle. Same ggplot2 + split–apply–combine habit as FuelEcon_R.
